# Monthly Youth Unemployment Statistics

## Dataset Description
Monthly unemployment statistics for **youth aged 15–24 and 15–30**, covering the age group that directly overlaps with fresh graduates entering the labour market. From the DOSM Labour Force Survey (LFS).

## Data Source
| Field | Detail |
|---|---|
| **Publisher** | Department of Statistics Malaysia (DOSM) |
| **Survey** | Labour Force Survey (LFS), monthly |
| **Catalogue** | https://open.dosm.gov.my/data-catalogue/lfs_month_youth |
| **Parquet URL** | `https://storage.dosm.gov.my/labour/lfs_month_youth.parquet` |
| **CSV URL** | `https://storage.dosm.gov.my/labour/lfs_month_youth.csv` |
| **License** | CC BY 4.0 — DOSM |

## Column Descriptions
| Column | Type | Description |
|---|---|---|
| `date` | date | Month (YYYY-MM-DD, DD always = 01) |
| `unemployed_15_24` | float | Number of unemployed persons aged 15–24 ('000) |
| `u_rate_15_24` | float | Unemployment rate for ages 15–24 (%) |
| `unemployed_15_30` | float | Number of unemployed persons aged 15–30 ('000) |
| `u_rate_15_30` | float | Unemployment rate for ages 15–30 (%) |

## Relevance to EduNilai
Youth unemployment directly affects the ROI calculation — a graduate who spends months unemployed after graduation earns nothing, increasing the effective cost of the degree. The 15–30 age band captures the extended transition period into graduate employment. The monthly series also clearly shows the COVID-19 shock (2020–2021) and recovery.

In [ ]:
# pip install pandas fastparquet
import pandas as pd
import matplotlib.pyplot as plt
import os

URL = 'https://storage.dosm.gov.my/labour/lfs_month_youth.parquet'
df = pd.read_parquet(URL)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nDate range:", df['date'].min(), "–", df['date'].max())
print("\nDescriptive stats:")
print(df[['u_rate_15_24', 'u_rate_15_30']].describe().round(2))

In [ ]:
df_2010 = df[df['date'].dt.year >= 2010].copy()

# Annual average for cleaner trend
df_annual = df_2010.set_index('date').resample('YE').mean(numeric_only=True).reset_index()

print("Annual average unemployment rates:")
print(df_annual[['date', 'u_rate_15_24', 'u_rate_15_30']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Unemployment rate trend — both age bands
axes[0].plot(df_2010['date'], df_2010['u_rate_15_24'], color='steelblue',
             linewidth=1, alpha=0.6, label='15–24 (monthly)')
axes[0].plot(df_2010['date'], df_2010['u_rate_15_30'], color='darkorange',
             linewidth=1, alpha=0.6, label='15–30 (monthly)')
axes[0].plot(df_annual['date'], df_annual['u_rate_15_24'], color='steelblue',
             linewidth=2, label='15–24 (annual avg)')
axes[0].plot(df_annual['date'], df_annual['u_rate_15_30'], color='darkorange',
             linewidth=2, label='15–30 (annual avg)')
axes[0].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'),
                alpha=0.12, color='red', label='COVID-19 period')
axes[0].set_title('Youth Unemployment Rate (%) — Monthly 2010 onwards')
axes[0].set_ylabel('Unemployment Rate (%)')
axes[0].legend(fontsize=8)

# Headcount trend
axes[1].plot(df_2010['date'], df_2010['unemployed_15_24'], color='steelblue',
             linewidth=1, label="15–24 ('000)")
axes[1].plot(df_2010['date'], df_2010['unemployed_15_30'], color='darkorange',
             linewidth=1, label="15–30 ('000)")
axes[1].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'),
                alpha=0.12, color='red')
axes[1].set_title("Youth Unemployed Persons ('000)")
axes[1].set_ylabel("'000 persons")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('../data/salary', exist_ok=True)
df_2010.to_csv('../data/salary/youth_unemployment_monthly.csv', index=False, encoding='utf-8')
print("Saved -> ../data/salary/youth_unemployment_monthly.csv")
print(f"Rows: {len(df_2010)}")